In [2]:
import json

json_path = r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\UniProtID_to_seq.json"

with open(json_path, "r") as f:
    id_to_seq = json.load(f)

P15289 = id_to_seq["P15289"]

In [1]:
import pandas as pd

# 원본 TSV 파일 경로
tsv_path = r"C:\Users\Kunny\Research\Project\BiConVarNet\ARSA\cagi7arsasample.tsv"

# 파일 읽기
df = pd.read_csv(tsv_path, sep='\t')

# 필요한 정보 파싱
def parse_aa_substitution(sub):
    # 예: "I25V" → WT="I", Pos=25, Mut="V"
    wt = sub[0]
    mut = sub[-1]
    pos = int(sub[1:-1]) - 2  # NP_000478.3 → P15289 보정
    return wt, pos, mut

# 결과 저장 리스트
records = []

for _, row in df.iterrows():
    aa_sub = row['aa_substitution']
    wt, mut_pos, mut = parse_aa_substitution(aa_sub)
    label = row['stability_score_48hr']

    records.append({
        "UniProtID": "P15289",
        "MutPos": mut_pos,
        "WT": wt,
        "Mut": mut,
        "Label": label,
        "StructureFile": "AF-P15289-F1-model_v4.pdb"
    })

# 결과 DataFrame 생성
output_df = pd.DataFrame(records)

In [2]:
output_df

,UniProtID,MutPos,WT,Mut,Label,StructureFile
0,P15289,23,I,V,0.647,AF-P15289-F1-model_v4.pdb
1,P15289,29,D,N,0.575,AF-P15289-F1-model_v4.pdb
2,P15289,29,D,E,0.533,AF-P15289-F1-model_v4.pdb
3,P15289,30,D,H,0.459,AF-P15289-F1-model_v4.pdb
4,P15289,31,L,P,0.599,AF-P15289-F1-model_v4.pdb
...,...,...,...,...,...,...
344,P15289,494,T,I,0.734,AF-P15289-F1-model_v4.pdb
345,P15289,496,R,L,0.718,AF-P15289-F1-model_v4.pdb
346,P15289,496,R,H,0.796,AF-P15289-F1-model_v4.pdb
347,P15289,496,R,P,0.416,AF-P15289-F1-model_v4.pdb


In [13]:
import os 

output_df.to_csv(r"C:\Users\Kunny\Research\Project\BiConVarNet\ARSA\sample_data.tsv", sep="\t", index=False)

In [1]:
import pandas as pd

# 입력/출력 경로
in_path = r"C:\Users\Kunny\Research\Project\BiConVarNet\ARSA\cagi7arsasubmissiontemplate_arsa.tsv"
out_path = r"C:\Users\Kunny\Research\Project\BiConVarNet\ARSA\ARSA_variants_formatted.tsv"

# 템플릿 읽기
df = pd.read_csv(in_path, sep="\t")

rows = []
for sub in df["aa_substitution"]:
    wt = sub[0]       # 첫 글자 → 원래 아미노산
    mut = sub[-1]     # 마지막 글자 → 변이 아미노산
    pos = int(sub[1:-1]) - 2   # 보정된 위치 (NP_000478.3 → P15289)

    rows.append({
        "UniProtID": "P15289",
        "MutPos": pos,
        "WT": wt,
        "Mut": mut,
        "Label": "NA",
        "StructureFile": "AF-P15289-F1-model_v4.pdb",
        "MutPos(pdb)": pos
    })

# 저장
out_df = pd.DataFrame(rows)
out_df.to_csv(out_path, sep="\t", index=False)

print(f"Saved → {out_path}")


Saved → C:\Users\Kunny\Research\Project\BiConVarNet\ARSA\ARSA_variants_formatted.tsv


In [ ]:
EVO="/home/kunny/EvoEF2/EvoEF2"
BASE=/mnt/c/Users/Kunny/Research/Dataset/Missense_Variant_dataset
INP=/mnt/c/Users/Kunny/Research/Dataset/Missense_Variant_dataset/alphafold_structures/AF-P15289-F1-model_v4.pdb
OUT=$BASE/evoef2_out_P15289

mkdir -p "$OUT/models"
cd "$OUT"
cp "$INP" ARSA.pdb

grep "^ATOM" ARSA.pdb | cut -c22 | sort -u

In [2]:
import os, csv

# === 사용자 경로 설정 ===
TSV_PATH = r"C:\Users\Kunny\Research\Project\BiConVarNet\ARSA\ARSA_variants_formatted.tsv"
# EvoEF2 작업 폴더(WSL에서 사용 중인 폴더와 같은 곳을 지정하면 편함)
OUT_DIR  = r"C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\evoef2_out_P15289"

# 필터 조건 (원하면 전체를 만들려면 조건을 바꾸거나 None로 둠)
TARGET_UID = "P15289"           
TARGET_PDB = "AF-P15289-F1-model_v4.pdb"  

os.makedirs(OUT_DIR, exist_ok=True)

mutlist_path = os.path.join(OUT_DIR, "mut_list_all.txt")          # EvoEF2 변이 리스트
mapping_path = os.path.join(OUT_DIR, "mutants_mapping.tsv")       # 참고용 매핑(모델링 후 매칭/디버깅 편함)

n = 0
with open(TSV_PATH, newline="", encoding="utf-8") as f, \
     open(mutlist_path, "w", encoding="ascii", newline="\n") as g, \
     open(mapping_path, "w", encoding="utf-8", newline="") as m:

    rdr = csv.DictReader(f, delimiter="\t")
    mw = csv.writer(m, delimiter="\t")
    mw.writerow(["UniProtID", "StructureFile", "MutPos", "WT", "Mut", "Label", "EvoEF2_mut_string"])

    for r in rdr:
        uid = r.get("UniProtID", "").strip()
        pdb = r.get("StructureFile", "").strip()
        if TARGET_UID and uid != TARGET_UID:
            continue
        if TARGET_PDB and pdb != TARGET_PDB:
            continue

        wt  = r.get("WT", "").strip().upper()
        mt  = r.get("Mut", "").strip().upper()
        pos = int(r.get("MutPos", "0"))

        # AlphaFold는 체인 A 가정
        evoef_mut = f"{wt}A{pos}{mt};"   # 예: GA123D;

        g.write(evoef_mut + "\n")

        mw.writerow([
            uid, pdb, pos, wt, mt, r.get("Label", ""), evoef_mut
        ])
        n += 1

print(f"[OK] 변이 {n}개를 '{mutlist_path}'에 작성했습니다.")
print(f"[OK] 매핑 파일: '{mapping_path}'")
print("\n다음 순서로 진행하세요 (WSL 쉘):")

[OK] 변이 8867개를 'C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\evoef2_out_P15289\mut_list_all.txt'에 작성했습니다.
[OK] 매핑 파일: 'C:\Users\Kunny\Research\Dataset\Missense_Variant_dataset\evoef2_out_P15289\mutants_mapping.tsv'

다음 순서로 진행하세요 (WSL 쉘):


In [ ]:
cd /mnt/c/Users/Kunny/Research/Dataset/Missense_Variant_dataset/evoef2_out_P15289
chmod +x run_evoef2_ddg_fold.sh

EVO="/home/kunny/EvoEF2/EvoEF2" \
WT_PDB="ARSA.pdb" \
MUT_LIST="mut_list_all.txt" \
bash ./run_evoef2_ddg_fold.sh